# C2 · Three outliers out of 303 points hijack the first component

**Type**: Break the model · **Level**: ●●○ independent · **Time**: about 40 minutes · **core path ★**

## Objective

Add three anomalous points to a cloud of 300 points and measure how much the first principal component rotates (by 90 degrees: it moves from one column to another). Fix with a clipping (winsorizing) of the extreme values and verify that
the direction goes back to the true one.

## Dataset

`A`: 300 points in 4 dimensions with standard deviations `[3, 2, 1, 0.7]` (the true first component is column 0). `B` = `A` plus three outliers that differ only in column 2 (values around ±58).

## Educational purpose

Understand that PCA is based on **variance**, hence on squares: a single very distant point has a disproportionate weight. The defect does not show in the accuracy of a model: it shows in the direction of the components and in the explained variance.

## Task

1. Compute `pc1_A` and `pc1_B` (first component of `PCA` on `A` and on `B`) and `A_column`, `B_column` (index of the column with the largest absolute loading in each). Save `angle_degrees` = angle between the two directions (`arccos(|pc1_A · pc1_B|)` in degrees).
2. Save `ratio_A` and `ratio_B` (share of variance of the first component before and after) and `outlier_share` (share of the 3 outliers over the 303 points, as a percentage).
3. **Fix:** clip every column of `B` between the 1st and the 99th percentile (`np.clip`, percentiles computed on the column) → `B_clip`. Save `clip_column` (column with the largest loading in PC1) and `clip_angle_degrees` (angle between the PC1 of `B_clip` and the PC1 of `A`).
4. Also check with the **Mahalanobis distances** that the outliers are identifiable: save `outlier_indices` (the three points of `B` with the largest Mahalanobis distance, computed with the covariance of `A`).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.decomposition import PCA

rng = np.random.default_rng(1)
A = rng.multivariate_normal([0, 0, 0, 0], np.diag([9, 4, 1, 0.5]), 300)
outliers = np.array([[0, 0, 60, 0], [0, 0, -55, 5], [0, 0, 58, -3]])
B = np.r_[A, outliers]
print(A.shape, B.shape)

## Your work

In [ ]:
pc1_A = None
pc1_B = None
A_column = None
B_column = None
angle_degrees = None
ratio_A = None
ratio_B = None
outlier_share = None

In [ ]:
B_clip = None
clip_column = None
clip_angle_degrees = None

In [ ]:
outlier_indices = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Angle: `np.degrees(np.arccos(min(1, abs(pc1_A @ pc1_B))))`. Clipping: `lo, hi = np.percentile(B, [1, 99], axis=0); B_clip = np.clip(B, lo, hi)`. Mahalanobis: `d = B - A.mean(0); m2 = np.einsum("ij,jk,ik->i", d, np.linalg.inv(np.cov(A.T)), d)`; `np.argsort(m2)[-3:]`.'):
    check(A_column == 0 and B_column == 2, "without outliers PC1 is column 0; with the three outliers it is column 2", "np.abs(pc1).argmax()")
    check(angle_degrees > 85 and close(outlier_share, 100 * 3 / 303), "PC1 rotates by about 90 degrees because of three points (1% of the data)", "arccos(|pc1_A . pc1_B|)")
    check(ratio_B > ratio_A and ratio_A < 0.65 and ratio_B > 0.7, "the variance explained by PC1 goes from 0.59 to 0.73: misleading", "explained_variance_ratio_[0]")
    check(B_clip.shape == B.shape and clip_column == 0 and clip_angle_degrees < 15, "after the clipping PC1 is column 0 again (angle with the true one below 15 degrees)", "np.clip(column, percentile 1, percentile 99) for every column")
    check(sorted(outlier_indices) == [300, 301, 302], "the Mahalanobis distance identifies the three outliers (indices 300, 301, 302)", "use the covariance of A and the mean of A")
done()

---
The commented solution is in `solutions/C_break_the_model/C2_three_outliers_hijack_pc1.ipynb`. Open it after you pass the check, or after you have really tried.